In [15]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

import numpy as np
import pandas as pd
from os import listdir
from os.path import isfile, join
from mhcnames import normalize_allele_name
import mhcnames
import mhcgnomes

METHOD = "hmm_pomegranate"

# 1. Read data

In [16]:
DATA_PATH_BASE = 'G:/My Drive/Work/ITMO/data'
DATA_PATH = f'{DATA_PATH_BASE}/length_dependent_hmm/' #Work laptop

# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu

In [18]:
onlyfiles

['data_A0201.csv',
 'data_B2705.csv',
 'data_B0702.csv',
 'data_A0301.csv',
 'data_A0101.csv',
 'data_DRB10101.csv',
 'data_DRB10401.csv',
 'data_DPA10103-DPB10401.csv',
 'data_DRB10301.csv',
 'data_DRB11501.csv',
 'data_DRB30101.csv',
 'data_DRB50101.csv',
 'data_A2402.csv',
 'data_A1101.csv',
 'data_DPA10202-DPB10501.csv',
 'data_IAb.csv',
 'data_IAd.csv',
 'data_IAk.csv',
 'data_IAq.csv',
 'data_IAs.csv',
 'data_IAu.csv',
 'data_IEd.csv',
 'data_H2-AAb-ABb.csv',
 'data_H2-AAd-ABd.csv',
 'data_H2-AAk-ABk.csv',
 'data_H2-AAq-ABq.csv',
 'data_H2-AAs-ABs.csv',
 'data_H2-AAu-ABu.csv',
 'data_H2-EAd-EBd.csv']

In [14]:

p1 = mhcgnomes.parse("H2_IEd")
print(p1)
print(p1.compact_string(include_species=True))
print(p1.to_string())


p2 = mhcgnomes.parse(p1.compact_string(include_species=True))
print(p2)
print(p2.compact_string())


Pair(alpha=Allele(gene=Gene(species=Species(name='Mus musculus', mhc_prefix='H2'), name='EA', mutations=()), allele_fields=('d',), annotations=(), mutations=()), beta=Allele(gene=Gene(species=Species(name='Mus musculus', mhc_prefix='H2'), name='EB', mutations=()), allele_fields=('d',), annotations=(), mutations=()), mhc_class='II')
H2-EAd-EBd
H2-EA*d/EB*d
Pair(alpha=Allele(gene=Gene(species=Species(name='Mus musculus', mhc_prefix='H2'), name='EA', mutations=()), allele_fields=('d',), annotations=(), mutations=()), beta=Allele(gene=Gene(species=Species(name='Mus musculus', mhc_prefix='H2'), name='EB', mutations=()), allele_fields=('d',), annotations=(), mutations=()), mhc_class='IIa')
EAd-EBd


In [17]:
onlyfiles = [f for f in listdir(DATA_PATH) if isfile(join(DATA_PATH, f))]
ALLELES = []
NOT_NORMALIZED = []
per_allele_train_df = dict()
for file_name in onlyfiles:
    allele_name = file_name.split("_", 1)[1].split('.')[0]
    NOT_NORMALIZED.append(allele_name)
    allele_name = mhcgnomes.parse(allele_name).to_string()
    per_allele_train_df[allele_name] =  pd.read_csv(DATA_PATH + file_name,sep=";")
    ALLELES.append(allele_name)
ALLELES

ParseError: Could not parse 'IAd'

In [4]:
mhcnames.compact_allele_name('HLA-DPA1*01:03-DPB1*04:01')

'DPA10103-DPB10401'

# 2. Make per length split

In [5]:
from sklearn.model_selection import KFold
MIN_LENGTH = 8
MAX_LENGTH = 30

#leave only binders
def getBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 1]['peptide'].values
    return result_peptides
def getNonBindersPeptidesOnly(df):
    result_peptides = df[df['binder'] == 0]['peptide'].values
    return result_peptides

def make_per_length_dict(df, binder_flag):
    lengths = set(df['length'])
    lengths = set([length for length in lengths if length >= MIN_LENGTH and length <= MAX_LENGTH])
    per_length_dict = dict()
    for length in lengths:
        result_peptides = df[(df['binder'] == binder_flag) & (df['length'] == length)]['peptide'].values
        per_length_dict[length] = result_peptides
    return per_length_dict

kf = KFold(n_splits=5, shuffle=True)
def make_per_length_per_KFold_dict(per_length_dict):

    per_length_per_kfold_peptides_train = dict()
    per_length_per_kfold_peptides_test = dict()

    print("Length \t Train/Test for splits", end="")
    for target_length in per_length_dict.keys():
        print(f"\n{target_length}:", end="\t")
        peptides_array = per_length_dict[target_length]
        per_length_per_kfold_peptides_train[target_length] = dict()
        per_length_per_kfold_peptides_test[target_length] = dict()
        if len(peptides_array) > kf.n_splits:
            i = 0
            for train_indexes_b, test_indexes_b in kf.split(peptides_array):
                print("{}/{}\t".format(len(train_indexes_b), len(test_indexes_b)), end="")
                current_train = peptides_array[train_indexes_b]
                current_test = peptides_array[test_indexes_b]
                per_length_per_kfold_peptides_train[target_length][i] = current_train
                per_length_per_kfold_peptides_test[target_length][i] = current_test
                i = i + 1
        else:

            print(f"Not enough peptides of length {target_length}", end="")
    print()
    return per_length_per_kfold_peptides_train , per_length_per_kfold_peptides_test





def addAuxColumns(per_allele_df):
    for allele_name, df in per_allele_df.items():
        df['length'] = df['peptide'].str.len()

per_allele_binders_train = dict()
per_allele_non_binders_train = dict()
per_allele_per_length_binders_train = dict()
per_allele_per_length_non_binders_train = dict()
per_allele_per_length_per_kfold_binders_train = dict()
per_allele_per_length_per_kfold_binders_test = dict()
per_allele_per_length_per_kfold_non_binders_train = dict()
per_allele_per_length_per_kfold_non_binders_test = dict()


for allele_name in ALLELES:
    print(f"Allele {allele_name}")
    df_train = per_allele_train_df[allele_name]
    df_train = df_train.sample(frac=1).reset_index(drop=True)
    per_allele_binders_train[allele_name] = getBindersPeptidesOnly(df_train)
    per_allele_non_binders_train[allele_name] = getNonBindersPeptidesOnly(df_train)
    per_allele_per_length_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=1)
    per_allele_per_length_non_binders_train[allele_name] = make_per_length_dict(df_train, binder_flag=0)
    print(f"Binders")
    per_allele_per_length_per_kfold_binders_train[allele_name],  per_allele_per_length_per_kfold_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_binders_train[allele_name])
    print(f"Non-binders")
    per_allele_per_length_per_kfold_non_binders_train[allele_name],  per_allele_per_length_per_kfold_non_binders_test[allele_name] = make_per_length_per_KFold_dict(per_allele_per_length_non_binders_train[allele_name])


Allele HLA-A*02:01
Binders
Length 	 Train/Test for splits
8:	2435/609	2435/609	2435/609	2435/609	2436/608	
9:	25678/6420	25678/6420	25678/6420	25679/6419	25679/6419	
10:	7080/1771	7081/1770	7081/1770	7081/1770	7081/1770	
11:	3576/895	3577/894	3577/894	3577/894	3577/894	
12:	1200/300	1200/300	1200/300	1200/300	1200/300	
13:	776/195	777/194	777/194	777/194	777/194	
14:	570/143	570/143	570/143	571/142	571/142	
15:	572/144	573/143	573/143	573/143	573/143	
16:	416/104	416/104	416/104	416/104	416/104	
17:	256/65	257/64	257/64	257/64	257/64	
18:	167/42	167/42	167/42	167/42	168/41	
19:	114/29	114/29	114/29	115/28	115/28	
20:	104/27	105/26	105/26	105/26	105/26	
21:	56/15	57/14	57/14	57/14	57/14	
22:	34/9	34/9	34/9	35/8	35/8	
23:	28/7	28/7	28/7	28/7	28/7	
24:	20/6	21/5	21/5	21/5	21/5	
25:	15/4	15/4	15/4	15/4	16/3	
26:	6/2	6/2	6/2	7/1	7/1	
27:	10/3	10/3	10/3	11/2	11/2	
28:	8/2	8/2	8/2	8/2	8/2	
29:	5/2	5/2	6/1	6/1	6/1	
30:	5/2	5/2	6/1	6/1	6/1	
Non-binders
Length 	 Train/Test for splits
8:	91/23	91

# 3. Save per_length_per_split to files

In [7]:
import os
from collections import defaultdict

TARGET_PATH_TO_FILES = DATA_PATH  + '/per_length_per_kfold_split/'
if not os.path.exists(TARGET_PATH_TO_FILES):
    os.makedirs(TARGET_PATH_TO_FILES)

def save_data(per_allele_per_length_per_kfold_data, data_type=None, dataset_type=None):
    assert data_type in ["binders", "nonbinders"]
    assert dataset_type in ["train", "test"]

    per_allele_dfs = defaultdict(list)
    for allele_name in per_allele_per_length_per_kfold_data.keys():
        for length, per_k_fold_dict in per_allele_per_length_per_kfold_data[allele_name].items():
                for split_num, peptides in per_k_fold_dict.items():
                    df = pd.DataFrame({"peptide":peptides, "length":length, "split": split_num})
                    per_allele_dfs[allele_name].append(df)
        total_df= pd.concat(per_allele_dfs[allele_name])
        target_allele_name = mhcnames.compact_allele_name(allele_name)
        total_df.to_csv(f'{TARGET_PATH_TO_FILES}{target_allele_name}_{data_type}_{dataset_type}.csv',sep=";", index=False)


save_data(per_allele_per_length_per_kfold_binders_train, "binders", "train")
# save_data(per_allele_per_length_per_kfold_non_binders_train, "nonbinders", "train")
save_data(per_allele_per_length_per_kfold_binders_test, "binders", "test")
# save_data(per_allele_per_length_per_kfold_non_binders_test, "nonbinders", "test")